In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

from options import parser
from utils.data_utils import build_generators, count_files, split_dataset
from utils.eda_utils import check_uniform_resolution, count_images_per_class, plot_class_distribution
from net.backbones import load_inception_backbone, load_resnet50_backbone, load_vgg16_backbone
from net.models import build_model1, build_model2, build_model3, build_model4, build_model5, build_model6
from train.common import train_and_save

opt = parser.parse_args([])  # notebook uses the CLI defaults from options.py

# Data Pre-Processing

## Checking Image Resolutions

In [ ]:
resolutions = check_uniform_resolution(opt.data_dir)
if len(resolutions) == 1:
    print(f"Uniform resolution: {resolutions.pop()}")
else:
    print(f"Non-uniform resolutions: {resolutions}")

## Counting and Plotting the Number of Images in Each Class

In [ ]:
class_counts = count_images_per_class(opt.data_dir, opt.classes)
plot_class_distribution(
    class_counts, save_path=os.path.join(opt.figures_dir, "class_distribution.png")
)

## Splitting the Dataset into Training and Validation Sets

In [ ]:
split_dataset(
    opt.data_dir, opt.classes, opt.train_dir, opt.val_dir,
    val_split=opt.val_split, seed=opt.seed,
)
print(f"train: {count_files(opt.train_dir)}  val: {count_files(opt.val_dir)}")

## Preparing Image Data Generators

In [ ]:
train_generator, validation_generator = build_generators(
    opt.train_dir, opt.val_dir, opt.img_size, opt.batch_size
)

# Model Training

## Model 1: Conv2D + MaxPooling

In [ ]:
model1 = build_model1(
    input_shape=(opt.img_size, opt.img_size, 3), num_classes=len(opt.classes)
)
model1.summary()

In [ ]:
history1 = train_and_save(
    model1, "model1_cnn", train_generator, validation_generator,
    opt.epochs, opt.ckpt_dir, opt.history_dir,
)

## Model 2: Transfer Learning (InceptionV3)

In [ ]:
inception_backbone = load_inception_backbone(
    opt.inception_weights, input_shape=(opt.img_size, opt.img_size, 3)
)
model2 = build_model2(inception_backbone, num_classes=len(opt.classes))
model2.build(input_shape=(None, opt.img_size, opt.img_size, 3))
model2.summary()

In [ ]:
history2 = train_and_save(
    model2, "model2_inception", train_generator, validation_generator,
    opt.epochs, opt.ckpt_dir, opt.history_dir,
)

## Model 3: Conv2D + LeakyReLU

In [ ]:
model3 = build_model3(
    input_shape=(opt.img_size, opt.img_size, 3), num_classes=len(opt.classes)
)
model3.summary()

In [ ]:
history3 = train_and_save(
    model3, "model3_leaky_cnn", train_generator, validation_generator,
    opt.epochs, opt.ckpt_dir, opt.history_dir,
)

## Model 4: Transfer Learning (InceptionV3) + LeakyReLU

In [ ]:
inception_backbone = load_inception_backbone(
    opt.inception_weights, input_shape=(opt.img_size, opt.img_size, 3)
)
model4 = build_model4(inception_backbone, num_classes=len(opt.classes))
model4.build(input_shape=(None, opt.img_size, opt.img_size, 3))
model4.summary()

In [ ]:
history4 = train_and_save(
    model4, "model4_inception_leaky", train_generator, validation_generator,
    opt.epochs, opt.ckpt_dir, opt.history_dir,
)

## Model 5: Transfer Learning (VGG16)

In [ ]:
vgg_backbone = load_vgg16_backbone(input_shape=(opt.img_size, opt.img_size, 3))
model5 = build_model5(vgg_backbone, num_classes=len(opt.classes))
model5.build(input_shape=(None, opt.img_size, opt.img_size, 3))
model5.summary()

In [ ]:
history5 = train_and_save(
    model5, "model5_vgg16", train_generator, validation_generator,
    opt.epochs, opt.ckpt_dir, opt.history_dir,
)

## Model 6: Transfer Learning (ResNet50)

In [ ]:
resnet_backbone = load_resnet50_backbone(input_shape=(opt.img_size, opt.img_size, 3))
model6 = build_model6(resnet_backbone, num_classes=len(opt.classes))
model6.build(input_shape=(None, opt.img_size, opt.img_size, 3))
model6.summary()

In [ ]:
history6 = train_and_save(
    model6, "model6_resnet50", train_generator, validation_generator,
    opt.epochs, opt.ckpt_dir, opt.history_dir,
)

# Evaluation

In [ ]:
import matplotlib.pyplot as plt

histories = {
    "Model 1": history1.history, "Model 2": history2.history,
    "Model 3": history3.history, "Model 4": history4.history,
    "Model 5": history5.history, "Model 6": history6.history,
}

fig, axs = plt.subplots(2, 2, figsize=(15, 12))
metrics = [
    ("accuracy", "Training Accuracy Comparison Over Epochs", "Accuracy", axs[0, 0]),
    ("val_accuracy", "Validation Accuracy Comparison Over Epochs", "Accuracy", axs[0, 1]),
    ("loss", "Training Loss Comparison Over Epochs", "Loss", axs[1, 0]),
    ("val_loss", "Validation Loss Comparison Over Epochs", "Loss", axs[1, 1]),
]
for key, title, ylabel, ax in metrics:
    for label, history in histories.items():
        ax.plot(history[key], label=label)
    ax.set_title(title)
    ax.set_xlabel("Epochs")
    ax.set_ylabel(ylabel)
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(opt.figures_dir, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()